# MP20 RCS Rat

## Create a Threshold Simulation Projection (800um spot)

In [ ]:
from run_stages.pattern_generation_stage import Text, Grating, Rectangle, Circle, FullField, Subframe, Frame, ProjectionSequence

# generate the pattern defined below
generate_pattern = True

# define projection parameters
duration = 10
on_duration = duration
intensity = 0.27
frequency = 2
switch_frequency = frequency
frame_width = 1125
conductivity = 1/2
device = 'MP20'
time_resolution = 0.5
extra_before_pulse = 0

# define the name for the projection sequence
name_template = "{}_RCS_800um_spot_{}Hz_{}ms_{}mW"

# initialize the list of projection sequences and their names
video_sequence_name = list()
list_projections = list()

# a subframe of a 800um spot followed by a subframe of darkness
subframes= [Subframe(duration_ms=duration, patterns=[Circle(position=(0, 0), unit="um", diameter=800)]),
            Subframe(duration_ms=((1/frequency)*1E3-duration), patterns=[FullField('black')])]

# create a frame from the given subframes
list_of_frames = [Frame(name=name_template.format(device, frequency, duration, intensity), repetitions=1, subframes=subframes)]
list_projections.append(ProjectionSequence(intensity_mW_mm2=intensity, frequency_Hz=frequency, frames=list_of_frames))

video_sequence_name.append(name_template.format(device, frequency, duration, intensity))

## Create an Acuity Simulation Projection (Alternating Gratings)


In [ ]:
from run_stages.pattern_generation_stage import Text, Grating, Rectangle, Circle, FullField, Subframe, Frame, ProjectionSequence

# create the projection pattern defined below
generate_pattern = True

# define general projection parameters
intensity = 1.25
time_resolution = 1
frame_width = 1125
conductivity = 1
device = 'MP20'

# define grating parameters
frequency = 64
duration = (1/frequency)*1E3
on_duration = duration/4
off_duration = (duration-on_duration)
extra_before_pulse = duration
grating_width = 50 # 137

# define switching parameters
switch_frequency = 2
repetitions = ((1/switch_frequency)*1E3//duration)

print(f"Each grating frame duration: {duration} ms, ON duration: {on_duration} ms, OFF duration: {off_duration} ms, repetitions: {repetitions}")
print("Intensity: {} mW/mm^2".format(intensity))
print("Device: {}".format(device))


# first grating
frame_name = f"RCS_{device}_{grating_width}um_grating_{frequency}Hz_{on_duration}ms_{intensity}mW_{time_resolution}ms_res"
subframes= [Subframe(duration_ms=on_duration, patterns=[Grating(position=(0, 0), rotation=0, unit="um", width_grating=grating_width, pitch_grating=grating_width)]),
            Subframe(duration_ms=off_duration, patterns=[FullField('black')])]
frames = [Frame(name=frame_name, repetitions=repetitions, subframes=subframes)]

# second grating
rever_frame_name = f"RCS_{device}_{grating_width}um_rever_grating_{frequency}Hz_{on_duration}ms_{intensity}mW_{time_resolution}ms_res"
subframes = [Subframe(duration_ms=on_duration, patterns=[Grating(position=(grating_width, 0), rotation=0, unit="um", width_grating=grating_width, pitch_grating=grating_width)]),
            Subframe(duration_ms=off_duration, patterns=[FullField('black')])]
frames.append(Frame(name=rever_frame_name, repetitions=repetitions, subframes=subframes))

list_projections = [ProjectionSequence(intensity_mW_mm2=intensity, frequency_Hz=frequency, frames=frames)]
video_sequence_name = [f"RCS_{device}_acuity_{grating_width}um_grating"]

## Define the Rest of the Simulation Configuration

In [ ]:
rpsim_config = {}

rpsim_config["model"] = "monopolar"
rpsim_config["pixel_size"] = 20
rpsim_config["pixel_size_suffix"] = ""
rpsim_config["frame_width"] = frame_width #750
rpsim_config["geometry"] = 'Flat_rat_RCS'
rpsim_config["number_of_diodes"] = 1
rpsim_config["sirof_capacitance"] = 6
rpsim_config["photosensitive_area_edge_to_edge"] = 16
rpsim_config["active_electrode_radius"] = 4.5
rpsim_config["light_to_current_conversion_rate"] = 0.5
rpsim_config["photosensitive_area"] = 158.085252133623

# R matrix parameters
rpsim_config["r_matrix_output_file"] = f'R_{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}.pkl'
rpsim_config["r_matrix_conductivity"] = conductivity

# dynamic simulation configuration
rpsim_config["Ipho_scaling"] = 1
rpsim_config["Isat"] = 0.3
rpsim_config["ideality_factor"] = 1.5
rpsim_config["shunt_resistance"] = None
rpsim_config["initial_Vactive"] = 0.4
rpsim_config["temperature"] = 37
rpsim_config["nominal_temperature"] = 25
rpsim_config["simulation_duration_sec"] = 1 / switch_frequency * 5
rpsim_config["simulation_resolution_ms"] = None

# input paths
rpsim_config["user_files_path"] = None
rpsim_config["pixel_label_input_file"] = f'image_sequence/pixel_label_PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}.pkl'

# Projection sequences related
rpsim_config["video_sequence_name"] = video_sequence_name

rpsim_config["pattern_generation"] = {"generate_pattern": generate_pattern}
add_projection_seq = any(generate_pattern) if type(generate_pattern) is list else generate_pattern
if add_projection_seq:
    tmp = \
        {
            "projection_sequences": list_projections,
            "font_path": None,
            "projection_sequences_stored_config": None
        }
    rpsim_config["pattern_generation"].update(tmp)

# define input files for monopolar arrays
rpsim_config["monopolar"] = \
    {
        "return_to_active_area_ratio": 4.0876,
        "r_matrix_simp_ratio": 0.1,#0.1,
        "r_matrix_input_file_px_pos": f'r_matrix/COMSOL_results/PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}_pos.csv',
        "r_matrix_input_file_active": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_active.csv',
        "r_matrix_input_file_EP_return_2D": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_return_2D-whole.csv',
        "r_matrix_input_file_diagonal": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_self.csv',
        "r_matrix_input_file_non_diagonal": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_Rmat.csv'
    }

# define input files for bipolar arrays
bipolar_dict = \
    {
        "additional_edges": 142,
        "r_matrix_simp_ratio": 0.1,
        "r_matrix_input_file_px_pos": f'r_matrix/COMSOL_results/PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}_pos.csv',
        "r_matrix_input_file_active": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_active.csv',
        "r_matrix_input_file_return": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return.csv',
        "r_matrix_input_file_return_neighbor": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return_neighbor.csv',
    }
if rpsim_config["model"] == 'bipolar':
    bipolar_dict["r_matrix_input_file_return_near"] = f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return_near.csv'

rpsim_config["bipolar"] = bipolar_dict

# post-process parameters
rpsim_config["post_process"] = \
    {
        "pulse_start_time_in_ms": (1 / switch_frequency) * 1e3 * 4,
        "pulse_duration_in_ms": on_duration,
        "average_over_pulse_duration": False,
        "pulse_extra_ms": extra_before_pulse,  
        "time_averaging_resolution_ms": time_resolution,
        "interpolation_resolution_ms": 1e-3,
        "multiprocessing": False,
        "cpu_to_use": None,
        "depth_values_in_um": None,
        "on_diode_threshold_mV": 50
    }

rpsim_config["plot_results"] = \
    {
        "plot_time_window_start_ms": (1 / switch_frequency) * 1e3 * 4,
        "plot_time_window_end_ms": duration,
        "plot_potential_depth_um": 5
    }

print(rpsim_config["post_process"]["pulse_start_time_in_ms"])
print(rpsim_config["post_process"]["pulse_duration_in_ms"])
print(rpsim_config["post_process"]["pulse_extra_ms"])

### EXECUTION
from RPSim import run_rpsim
# Stages name: "pattern_generation" - "resistive_mesh" - "current_sequence" - "circuit" - "simulation" - "post_process" - "plot_results"
run_stages = [ "pattern_generation", "resistive_mesh", "current_sequence", "circuit", "simulation", 'post_process', 'plot_results']
run_rpsim(configuration=rpsim_config, run_stages=run_stages) 


# PRIMA100 RCS Rat


## Create a Threshold Simulation Projection (800um spot)


In [ ]:
from run_stages.pattern_generation_stage import Text, Grating, Rectangle, Circle, FullField, Subframe, Frame, ProjectionSequence

# generate the pattern defined below
generate_pattern = True

# define projection parameters
duration = 10
on_duration = duration
intensity = 2.11
frequency = 2
switch_frequency = frequency
frame_width = 1125
conductivity = 1/2
device = 'PRIMA100'
time_resolution = 0.5
extra_before_pulse = 0


# define the name for the projection sequence
name_template = "{}_RCS_800um_spot_{}Hz_{}ms_{}mW"

# initialize the list of projection sequences and their names
video_sequence_name = list()
list_projections = list()

# a subframe of a 800um spot followed by a subframe of darkness
subframes= [Subframe(duration_ms=duration, patterns=[Circle(position=(0, 0), unit="um", diameter=800)]),
            Subframe(duration_ms=((1/frequency)*1E3-duration), patterns=[FullField('black')])]

# create a frame from the given subframes
list_of_frames = [Frame(name=name_template.format(device, frequency, duration, intensity), repetitions=1, subframes=subframes)]
list_projections.append(ProjectionSequence(intensity_mW_mm2=intensity, frequency_Hz=frequency, frames=list_of_frames))

video_sequence_name.append(name_template.format(device, frequency, duration, intensity))

## Create an Acuity Simulation Projection (Alternating Gratings)


In [ ]:
from run_stages.pattern_generation_stage import Text, Grating, Rectangle, Circle, FullField, Subframe, Frame, ProjectionSequence

# create the projection pattern defined below
generate_pattern = True

# define general projection parameters
intensity = 1.25
time_resolution = 1
frame_width = 1125
conductivity = 1
device = 'PRIMA100'

# define grating parameters
frequency = 64
duration = (1/frequency)*1E3
on_duration = duration/4
off_duration = (duration-on_duration)
extra_before_pulse = duration
grating_width = 50 # 137

# define switching parameters
switch_frequency = 2
repetitions = ((1/switch_frequency)*1E3//duration)

print(f"Each grating frame duration: {duration} ms, ON duration: {on_duration} ms, OFF duration: {off_duration} ms, repetitions: {repetitions}")
print("Intensity: {} mW/mm^2".format(intensity))
print("Device: {}".format(device))


# first grating
frame_name = f"RCS_{device}_{grating_width}um_grating_{frequency}Hz_{on_duration}ms_{intensity}mW_{time_resolution}ms_res"
subframes= [Subframe(duration_ms=on_duration, patterns=[Grating(position=(0, 0), rotation=0, unit="um", width_grating=grating_width, pitch_grating=grating_width)]),
            Subframe(duration_ms=off_duration, patterns=[FullField('black')])]
frames = [Frame(name=frame_name, repetitions=repetitions, subframes=subframes)]

# second grating
rever_frame_name = f"RCS_{device}_{grating_width}um_rever_grating_{frequency}Hz_{on_duration}ms_{intensity}mW_{time_resolution}ms_res"
subframes = [Subframe(duration_ms=on_duration, patterns=[Grating(position=(grating_width, 0), rotation=0, unit="um", width_grating=grating_width, pitch_grating=grating_width)]),
            Subframe(duration_ms=off_duration, patterns=[FullField('black')])]
frames.append(Frame(name=rever_frame_name, repetitions=repetitions, subframes=subframes))

list_projections = [ProjectionSequence(intensity_mW_mm2=intensity, frequency_Hz=frequency, frames=frames)]
video_sequence_name = [f"RCS_{device}_acuity_{grating_width}um_grating"]

## Define the Rest of the Simulation Configuration

In [ ]:
rpsim_config = {}

# geometry-defined configuration
rpsim_config["model"] = "bipolar"
rpsim_config["pixel_size"] = 100
rpsim_config["pixel_size_suffix"] = ""
rpsim_config["frame_width"] = frame_width #750
rpsim_config["geometry"] = 'Flat_rat_RCS'
rpsim_config["number_of_diodes"] = 2
rpsim_config["sirof_capacitance"] = 6
rpsim_config["photosensitive_area_edge_to_edge"] = 92
rpsim_config["active_electrode_radius"] = 17
rpsim_config["light_to_current_conversion_rate"] = 0.4
rpsim_config["photosensitive_area"] = 4075.72

# R matrix parameters
rpsim_config["r_matrix_output_file"] = f'R_{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}.pkl'
rpsim_config["r_matrix_conductivity"] = conductivity

# dynamic simulation configuration
rpsim_config["Ipho_scaling"] = 1
rpsim_config["Isat"] = 0.02
rpsim_config["ideality_factor"] = 1.14
rpsim_config["shunt_resistance"] = 720000.0
# shunt
rpsim_config["initial_Vactive"] = 0
rpsim_config["temperature"] = 37
rpsim_config["nominal_temperature"] = 25
rpsim_config["simulation_duration_sec"] = 1 / switch_frequency * 6
rpsim_config["simulation_resolution_ms"] = None

# input paths
rpsim_config["user_files_path"] = None
rpsim_config["pixel_label_input_file"] = f'image_sequence/pixel_label_PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}.pkl'

# projection sequences related
rpsim_config["video_sequence_name"] = video_sequence_name
rpsim_config["pattern_generation"] = {"generate_pattern": generate_pattern}
if generate_pattern:
    tmp = \
        {
            "projection_sequences": list_projections,
            "font_path": None,  # If set to None for, defaults to optometrist font Sloan.otf
            "projection_sequences_stored_config": None
            # Used for storing the config, but part of the skipped parameters
        }
    rpsim_config["pattern_generation"].update(tmp)

# define input files for monopolar arrays
rpsim_config["monopolar"] = \
    {
        "return_to_active_area_ratio": 5.7525,  # ratio between return area and total active area
        "r_matrix_simp_ratio": 0.1,
        "r_matrix_input_file_px_pos": f'r_matrix/COMSOL_results/PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}_pos.csv',
        "r_matrix_input_file_active": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_active.csv',
        "r_matrix_input_file_EP_return_2D": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_return_2D-whole.csv',
        "r_matrix_input_file_diagonal": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_self.csv',
        # used for resistive mesh only
        "r_matrix_input_file_non_diagonal": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_EP_Rmat.csv'
        # used for resistive mesh only
    }

# define input files for bipolar arrays
bipolar_dict = \
    {
        "additional_edges": 104,
        "r_matrix_simp_ratio": 0.1,
        "r_matrix_input_file_px_pos": f'r_matrix/COMSOL_results/PS{rpsim_config["pixel_size"]}{rpsim_config["pixel_size_suffix"]}_pos.csv',
        "r_matrix_input_file_active": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_active.csv',
        "r_matrix_input_file_return": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return.csv',
        "r_matrix_input_file_return_neighbor": f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return_neighbor.csv',
        # used for resistive mesh only
    }
if rpsim_config["model"] == 'bipolar':  # Special file existing only for the bipolar PS100 and PS75 configurations
    bipolar_dict["r_matrix_input_file_return_near"] = f'r_matrix/COMSOL_results/{rpsim_config["geometry"]}/{rpsim_config["geometry"]}_PS{rpsim_config["pixel_size"]}_UCD_return_near.csv'

rpsim_config["bipolar"] = bipolar_dict

# post-process parameters
rpsim_config["post_process"] = \
    {
        "pulse_start_time_in_ms": (1 / switch_frequency) * 1e3 * 5,
        "pulse_duration_in_ms": on_duration,
        "average_over_pulse_duration": False,
        "pulse_extra_ms": extra_before_pulse,  
        "time_averaging_resolution_ms": time_resolution,
        "interpolation_resolution_ms": 1e-3,
        "multiprocessing": False,
        "cpu_to_use": None,
        "depth_values_in_um": None,
        "on_diode_threshold_mV": 50
    }

rpsim_config["plot_results"] = \
    {
        "plot_time_window_start_ms": (1 / switch_frequency) * 1e3 * 5,
        "plot_time_window_end_ms": duration,
        "plot_potential_depth_um": 75
    }

print(rpsim_config["post_process"]["pulse_start_time_in_ms"])
print(rpsim_config["post_process"]["pulse_duration_in_ms"])
print(rpsim_config["post_process"]["pulse_extra_ms"])

### EXECUTION
from RPSim import run_rpsim
# Stages name: "pattern_generation" - "resistive_mesh" - "current_sequence" - "circuit" - "simulation" - "post_process" - "plot_results"
run_stages = [ "pattern_generation", "resistive_mesh", "current_sequence", "circuit", "simulation", 'post_process', 'plot_results']
run_rpsim(configuration=rpsim_config, run_stages=run_stages) 